# Week 6 — Constraint Satisfaction: Inference Beats Search

**Lesson plan:** [`../weeks/week-06.md`](../weeks/week-06.md) · **Slides:** [`../slides/week-06/deck.md`](../slides/week-06/deck.md)

Live coding, 25 minutes. Build a CSP solver, then **ablate** it — the artifact of
this notebook is a table showing three orders of magnitude gained from about thirty
lines of inference and ordering.

> **Same total search algorithm throughout.** All of the gain comes from inference
> and ordering, not from searching harder. Students who internalize this understand
> why "just use a bigger model" is not always the answer.

## 1 · The CSP: ⟨X, D, C⟩

Variables, domains, constraints. Sudoku as the running example — 81 variables,
domains 1–9, and 27 all-different constraints.

In [ ]:
from collections import deque
import time
import itertools


class CSP:
    """Binary-constraint CSP with explicit neighbour lists."""

    def __init__(self, variables, domains, neighbors):
        self.variables = list(variables)
        self.domains = {v: set(domains[v]) for v in variables}
        self.neighbors = neighbors          # var -> set of conflicting vars
        self.pruned = []                    # trail for undo

    def consistent(self, var, value, assignment):
        """All-different: no neighbour already holds this value."""
        return all(assignment.get(n) != value for n in self.neighbors[var])

    def degree(self, var):
        return len(self.neighbors[var])


def sudoku_csp(grid):
    """grid: 81-char string, '.' or '0' for blank."""
    cells = [(r, c) for r in range(9) for c in range(9)]
    neighbors = {}
    for (r, c) in cells:
        peers = set()
        for k in range(9):
            peers.add((r, k))
            peers.add((k, c))
        br, bc = 3 * (r // 3), 3 * (c // 3)
        for dr in range(3):
            for dc in range(3):
                peers.add((br + dr, bc + dc))
        peers.discard((r, c))
        neighbors[(r, c)] = peers

    domains = {}
    for i, (r, c) in enumerate(cells):
        ch = grid[i]
        domains[(r, c)] = {int(ch)} if ch not in ".0" else set(range(1, 10))
    return CSP(cells, domains, neighbors)


def show_sudoku(assignment):
    for r in range(9):
        row = " ".join(str(assignment.get((r, c), ".")) for c in range(9))
        print(row[:5] + "|" + row[6:11] + "|" + row[12:])
        if r in (2, 5):
            print("-" * 5 + "+" + "-" * 5 + "+" + "-" * 5)


# A genuinely hard instance -- "Platinum Blonde"-class, designed to defeat
# naive backtracking.
HARD = ("..............3.85..1.2.......5.7.....4...1...9.......5"
        "......73..2.1........4...9")
EASY = ("53..7....6..195....98....6.8...6...34..8.3..17...2...6"
        ".6....28....419..5....8..79")
MED  = ("4.....8.5.3..........7......2.....6.....8.4......1...."
        "...6.3.7.5..2.....1.4......")

csp = sudoku_csp(HARD)
print("blank cells:", sum(1 for v in csp.variables if len(csp.domains[v]) > 1))

## 2 · Backtracking, with switches for each technique

One function. Four booleans. That is the ablation harness.

In [ ]:
def backtrack_search(csp, use_fc=False, use_mrv=False, use_lcv=False,
                     use_ac3=False, limit=2_000_000):
    counter = {"calls": 0}
    domains = {v: set(csp.domains[v]) for v in csp.variables}

    if use_ac3:
        if not ac3(csp, domains):
            return None, counter

    assignment = {v: next(iter(domains[v])) for v in csp.variables
                  if len(domains[v]) == 1}

    # Propagate the GIVENS before search starts. Without this the domains of
    # blank cells stay full, MRV has nothing to discriminate on, and the whole
    # ablation collapses. (This bug is easy to write and hard to spot -- the
    # solver still returns correct answers, just far too slowly.)
    if use_fc:
        for v, val in assignment.items():
            for n in csp.neighbors[v]:
                if n not in assignment:
                    domains[n].discard(val)

    def select_unassigned():
        unassigned = [v for v in csp.variables if v not in assignment]
        if not use_mrv:
            return unassigned[0]
        return min(unassigned, key=lambda v: (len(domains[v]), -csp.degree(v)))

    def order_values(var):
        vals = list(domains[var])
        if not use_lcv:
            return vals
        # LCV: prefer the value that rules out fewest neighbour options
        def conflicts(val):
            return sum(1 for n in csp.neighbors[var]
                       if n not in assignment and val in domains[n])
        return sorted(vals, key=conflicts)

    def forward_check(var, value):
        """Remove value from unassigned neighbours. Returns trail or None."""
        removed = []
        for n in csp.neighbors[var]:
            if n not in assignment and value in domains[n]:
                domains[n].discard(value)
                removed.append((n, value))
                if not domains[n]:
                    return removed, True      # domain wiped out -> fail
        return removed, False

    def undo(removed):
        for (n, value) in removed:
            domains[n].add(value)

    def recurse():
        counter["calls"] += 1
        if counter["calls"] > limit:
            raise TimeoutError("call limit exceeded")
        if len(assignment) == len(csp.variables):
            return dict(assignment)

        var = select_unassigned()
        for value in order_values(var):
            if csp.consistent(var, value, assignment):
                assignment[var] = value
                removed, wiped = ([], False)
                if use_fc:
                    removed, wiped = forward_check(var, value)
                if not wiped:
                    result = recurse()
                    if result is not None:
                        return result
                if use_fc:
                    undo(removed)
                del assignment[var]
        return None

    try:
        return recurse(), counter
    except TimeoutError:
        return "TIMEOUT", counter

## 3 · AC-3 — arc consistency as preprocessing

Enforce that for every value of `X` there exists *some* compatible value of `Y`,
for every constrained pair. It runs to a fixpoint before search begins.

In [ ]:
def ac3(csp, domains):
    """Returns False if an inconsistency is found. Mutates `domains`."""
    queue = deque((x, y) for x in csp.variables for y in csp.neighbors[x])

    def revise(x, y):
        removed = False
        for vx in set(domains[x]):
            # all-different: vx survives iff y has some value != vx
            if not any(vy != vx for vy in domains[y]):
                domains[x].discard(vx)
                removed = True
        return removed

    while queue:
        x, y = queue.popleft()
        if revise(x, y):
            if not domains[x]:
                return False
            for z in csp.neighbors[x]:
                if z != y:
                    queue.append((z, x))
    return True


d = {v: set(sudoku_csp(HARD).domains[v]) for v in sudoku_csp(HARD).variables}
c = sudoku_csp(HARD)
before = sum(len(x) for x in c.domains.values())
ac3(c, d)
after = sum(len(x) for x in d.values())
print(f"total domain values before AC-3: {before}")
print(f"                  after  AC-3: {after}   ({before-after} eliminated)")

## 4 · ⚠️ The ablation table — the artifact of this session

Same search algorithm, four configurations, one hard puzzle.

The plain-backtracking row has a call limit so the notebook finishes; in a live
session, run it uncapped and let it hang for fifteen seconds before interrupting.
The hang *is* the lesson.

In [ ]:
configs = [
    ("plain backtracking",        dict()),
    ("+ forward checking",        dict(use_fc=True)),
    ("+ MRV",                     dict(use_fc=True, use_mrv=True)),
    ("+ LCV",                     dict(use_fc=True, use_mrv=True, use_lcv=True)),
    ("+ AC-3 preprocessing",      dict(use_fc=True, use_mrv=True, use_ac3=True)),
]

LIMIT = 300_000

for name, grid in [("EASY (30 givens)", EASY), ("MEDIUM (24 givens)", MED),
                   ("HARD (17 givens)", HARD)]:
    print(f"\n{name}")
    print(f"  {'configuration':<24}{'calls':>12}{'time':>10}{'solved':>9}")
    print("  " + "-" * 55)
    for label, kw in configs:
        csp = sudoku_csp(grid)
        t0 = time.perf_counter()
        result, counter = backtrack_search(csp, limit=LIMIT, **kw)
        dt = time.perf_counter() - t0
        status = "TIMEOUT" if result == "TIMEOUT" else ("yes" if result else "no")
        print(f"  {label:<24}{counter['calls']:>12,}{dt:>9.3f}s{status:>9}")

print(f"\n(TIMEOUT = exceeded {LIMIT:,} backtrack calls. In a live session, run"
      f"\n plain backtracking uncapped on HARD and let it hang. The hang is the lesson.)")

### What the table says

Read the **EASY** row first: `4,209 -> 260 -> 52 -> 1` calls. Then note that on
**HARD**, plain backtracking and forward checking *both* time out, and MRV is what
makes the puzzle solvable at all.

- **Forward checking** is a large win on easy and medium instances: detecting a
  wiped-out domain the moment it happens beats discovering the failure ten levels
  deeper. On HARD it is not enough by itself.
- **MRV** is the single biggest jump, and on HARD it is the difference between
  *solved* and *never finishes*. Its logic is worth stating precisely:

  > **Variable ordering wants to fail fast. Value ordering wants to succeed
  > first.**

  MRV picks the *most* constrained variable — if this branch is doomed, find out
  now with the fewest possible children. LCV picks the *least* constraining value —
  if this branch can work, take the path most likely to work.

  The asymmetry confuses students every year. Name it explicitly.
- **AC-3** front-loads the propagation. On EASY it solves the grid outright — **1
  backtrack call**, meaning the search never had to guess.

> **Three to four orders of magnitude, from ~30 lines that add no search power at
> all.**

### ⚠️ Two honest observations from this data

**LCV is not free, and it is not always a win.** On MEDIUM it helps (375 → 108
calls). On HARD it *hurts* (34,478 → 56,813). Value ordering is a heuristic gamble:
you pay to compute it on every branch and it does not always pay back. Report what
you measure, not what the textbook implies.

**AC-3 added nothing over MRV on MEDIUM and HARD** (identical call counts). Its
preprocessing had already been subsumed by forward checking plus good ordering.
Techniques are not additive — a fact that only an ablation reveals, which is why
you run one instead of stacking every trick you know.

In [ ]:
csp = sudoku_csp(HARD)
sol, counter = backtrack_search(csp, use_fc=True, use_mrv=True, use_ac3=True)
print(f"solved in {counter['calls']:,} backtrack calls\n")
show_sudoku(sol)

# Verify the solution is actually valid -- never trust a solver you just wrote.
def valid(sol):
    for var, val in sol.items():
        for n in csp.neighbors[var]:
            if sol[n] == val:
                return False
    return len(sol) == 81 and all(1 <= v <= 9 for v in sol.values())

print("\nvalid solution:", valid(sol))
assert valid(sol)

## 5 · The same solver on a different problem

A CSP solver is **problem-independent** — that is the entire point of the
representation. Map colouring, no changes to the solver.

In [ ]:
AUS_NEIGHBORS = {
    "WA": {"NT", "SA"}, "NT": {"WA", "SA", "Q"}, "SA": {"WA", "NT", "Q", "NSW", "V"},
    "Q": {"NT", "SA", "NSW"}, "NSW": {"Q", "SA", "V"}, "V": {"SA", "NSW", "T"},
    "T": {"V"},
}
COLORS = {"red", "green", "blue"}

aus = CSP(list(AUS_NEIGHBORS), {v: COLORS for v in AUS_NEIGHBORS}, AUS_NEIGHBORS)
sol, counter = backtrack_search(aus, use_fc=True, use_mrv=True)
print(f"map colouring solved in {counter['calls']} calls:")
for k, v in sorted(sol.items()):
    print(f"  {k:<4} {v}")

# Now make it unsatisfiable: 2 colours cannot 3-colour a triangle.
aus2 = CSP(list(AUS_NEIGHBORS), {v: {"red", "green"} for v in AUS_NEIGHBORS},
           AUS_NEIGHBORS)
sol2, counter2 = backtrack_search(aus2, use_fc=True, use_mrv=True)
print(f"\nwith 2 colours: {sol2} after {counter2['calls']} calls")
print("""
The solver PROVED no 2-colouring exists by exhausting the space. Exhaustion is a
proof -- and it is a categorical capability that reappears in week 9's planner and
has no LLM analogue. Note it for the scorecard.""")

## 6 · The modern counterpart — Logic-LM

Pan et al. (2023). The architecture for the studio:

```
natural language  --LLM-->  formal CSP  --YOUR SOLVER-->  answer
                                              |
                                        (sound, complete)
```

The LLM never solves anything. It **translates**. Your solver, which you can prove
correct, does the reasoning.

## Studio — three arms

| Arm | Pipeline |
|---|---|
| **A** | hand-encode the puzzle → your solver |
| **B** | LLM writes the CSP encoding → your solver |
| **C** | LLM solves the puzzle directly |

**The failure mode moves, and that is the finding:** arm C produces *wrong
answers*; arm B produces *malformed encodings* — which your solver **rejects
loudly** instead of answering wrongly.

> A system that fails loudly is worth more than one that fails quietly at the same
> rate. Put that on the scorecard under soundness, not accuracy.